# 턱걸이 솔루션

1. 턱이 바 위에 충분히 올라 왔는가?
2. 팔꿈치가 펴졌는가?
3. 몸의 반동을 주면서 올라가는가?

1. 팔이 쭉 펴져야 한다
- 발의 좌표가 가장 낮은 최저점 기준으로 판단
- 손목-팔꿈치-어깨의 각도를 몇도로 하는걸 펴졌다고?

2. 몸을 사용하지 말아야 한다

3. 무릎을 굽히지 말아야 한다

4. 다리를 벌리지 말아야 한다.
- 고관절 사이의 중점을 기준으로 왼 무릎과 오른 무릎의 각도를 측정하여 판단

5. 위 조건을 모두 만족 시 턱이 봉을 넘었을 때 +1

- 턱의 위치는 눈-코의 거리에 비례하여 예측
    
    > 일반적인 사람의 눈\~코 : 코\~턱 길이 비율 사용
    턱을 들 수 있기 때문에 귀\~코(y값) : 코\~턱 길이 비율을 사용하면 좋을 듯!!

- 봉의 위치는 손목-팔꿈치의 거리에 비례해 예측
    
    > 일반적인 사람의 손크기 : 전완의 비율 사용

- 예측한 턱의 y좌표가 봉의 y좌표를 넘었을 때 +1

기본 영상 이미지 : 720x1280

In [4]:
import numpy as np

<img src="hrnet_pose.png" width = "130px" height="200px" alt="error"></img>

In [5]:
img_size = [720, 1280]

In [6]:
# HRNET을 거친 좌표 예시
coordinates = [[301.91, 491.14, 0.98145],
  [      311.4,       476.9,      0.9585],
  [     287.67,      481.65,     0.97852],
  [     339.88,      491.14,     0.97461],
  [     278.18,      505.38,     0.96631],
  [     406.33,      562.33,     0.92285],
  [     273.44,     595.55,     0.91162],
  [     491.76,      609.79,     0.95947],
  [      197.5,      657.25,     0.96387],
  [     505.99,      524.36,     0.98438],
  [     178.51,      576.57,     0.89844],
  [      382.6,      780.65,     0.89648],
  [     297.17,       775.9,     0.89893],
  [     425.31,      932.53,     0.92432],
  [     254.45,      946.76,     0.91699],
  [     463.28,      1108.1,      0.9209],
  [     225.97,      1108.1,     0.90088]]

In [7]:
IsArmStretch = 0 # 팔은 쭉 폈는가
IsBodyStop = 0 # 몸을 사용하지 않았는가
IsKneeStop = 0 # 무릎을 굽히지 않았는가
IsLegClose = 0 # 다리를 벌리지 않았는가
IsChinOver = 0 # 턱이 봉을 넘었는가

1. 팔이 쭉 펴져야 한다

* 손목 - 목 - 손목 각도  
* 어깨 - 팔꿈치 - 손목 각도

언제 체크할건데? : 가장 낮은 발의 y축 좌표가 이전 2번의 프레임보다 모두 높아졌을 때?

오른팔 : 5, 7, 9  
왼팔 : 6, 8, 10

In [5]:
# 넘어야 함
arm_angle = int(input('팔 각도를 입력해 주세요 : '))

팔 각도를 입력해 주세요 : 130


In [8]:
def calculate_angle2D_1(a, b, c, direction=-1):
    """
    calculate_angle2D is divided by left and right side because this function uses external product
    input : a,b,c -> landmarks with shape [x,y,z,visibility]
          direction -> int -1 or 1
                      -1 means Video(photo) for a person's left side and 1 means Video(photo) for a person's right side
    output : angle between vector ba and bc with range 0~360
    """
    # external product's z value
    external_z = (b[0]-a[0])*(b[1]-c[1]) - (b[1]-a[1])*(b[0]-c[0])

    a = np.array(a[:2]) #first
    b = np.array(b[:2]) #mid
    c = np.array(c[:2]) #end

    ba = b-a
    bc = b-c
    dot_result = np.dot(ba, bc)


    ba_size = np.linalg.norm(ba)
    bc_size = np.linalg.norm(bc)
    radi = np.arccos(dot_result / (ba_size*bc_size))
    angle = np.abs(radi*180.0/np.pi)

    
    if external_z * direction > 0:
        angle = 360 - angle
    
    return round(angle, 2)

In [62]:
# 팔 관절값
joint_idx = {'right_elbow':[6,8,10], 'left_elbow':[5, 7, 9]}

In [73]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid = coordinates[v[1]]
    end = coordinates[v[2]]
    
    if 'right' in k:
        dimension = 1
    else:
        dimension = -1
    
    angle_tmp = calculate_angle2D_1(first ,mid, end, dimension)
    print(angle_tmp)
    # 각도가 지정한 각도를 넘었다면
    if angle_tmp >= arm_angle:
        IsArmStretch = 1
    else:
        IsArmStretch = 0
        break

64.15


In [68]:
IsArmStretch

0

2. 다리를 벌리지 말아야 한다.

* 고관절-무릎-발목 사이의 각도 -> 개구리를 잡고
* 무릎의 y값과 발목의 y값 -> 무릎이 구부러졌다고 판단
* 배꼽과 두 무릎 사이의 각도

언제 측정할 건데? : 계속  
가장 낮은 발의 y축 좌표가 이전 2번의 프레임보다 모두 높아졌을 때 기준 n번 이상 경고가 나오면 IsLegClose를 0으로

In [9]:
def calculate_angle2D_4(a, b1, b2, c, direction=-1):
    """
    calculate_angle2D is divided by left and right side because this function uses external product
    input : a,b,c -> landmarks with shape [x,y,z,visibility]
          direction -> int -1 or 1
                      -1 means Video(photo) for a person's left side and 1 means Video(photo) for a person's right side
    output : angle between vector ba and bc with range 0~360
    """
    b1 = np.array(b1[:2])
    b2 = np.array(b2[:2])
    
    b = [round(abs((b1[0]+b2[0])/2),2), round(abs((b1[1]+b2[1])/2),2)]
    
    # external product's z value
    external_z = (b[0]-a[0])*(b[1]-c[1]) - (b[1]-a[1])*(b[0]-c[0])

    a = np.array(a[:2]) #first
    b = np.array(b[:2]) #mid
    c = np.array(c[:2]) #end

    ba = b-a
    bc = b-c
    dot_result = np.dot(ba, bc)


    ba_size = np.linalg.norm(ba)
    bc_size = np.linalg.norm(bc)
    radi = np.arccos(dot_result / (ba_size*bc_size))
    angle = np.abs(radi*180.0/np.pi)

    
    if external_z * direction > 0:
        angle = 360 - angle
    
    return round(angle,2)

In [37]:
# 넘으면 안됨
leg_angle = int(input('다리 각도를 입력해 주세요 : '))

다리 각도를 입력해 주세요 : 70


In [48]:
# 다리 관절값
joint_idx = {'leg_between':[14, 12, 11, 13]} 

In [55]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid1 = coordinates[v[1]]
    mid2 = coordinates[v[2]]
    end = coordinates[v[3]]
   
    angle_tmp = calculate_angle2D_4(first ,mid1, mid2, end, 1)
    print(angle_tmp)
    # 각도가 지정한 각도를 넘지 않았다면
    if angle_tmp <= arm_angle:
        IsLegClose = 1
    else:
        IsLegClose = 0
        break

55.87


In [56]:
IsLegClose

1

5. 위 조건을 모두 만족 시 턱이 봉을 넘었을 때 +1

<img src="hrnet_pose.png" width = "130px" height="200px" alt="error"></img>

#### 예측 봉 위치

#### 예측 턱 위치
귀\~코 : 코\~턱 = 0.5 : 0.8  
코\~턱  = 1.6 \* 귀\~코

In [10]:
joint_idx = {'left_ear_nose':[3, 0], 'right_ear_nose':[4,0]} 

In [15]:
def calculate_chin(a, b):
    chin_x = b[0]
    a = np.array(a[1]) # ear,  y 좌표
    b = np.array(b[1]) # nose, y 좌표
    
    ear_to_nose = abs(a-b)
    # x좌표는 nose, y좌표는 nose + 
    chin = [chin_x, b+round(ear_to_nose*1.6,2)]
    
    return chin

In [16]:
for k, v in joint_idx.items():
    ear = coordinates[v[0]]
    nose = coordinates[v[1]]
    print(ear, nose)
    expected_chin = calculate_chin(ear ,nose)
    print('예측한 턱의 좌표 : ', expected_chin)

[339.88, 491.14, 0.97461] [301.91, 491.14, 0.98145]
예측한 턱의 좌표 :  [301.91, 491.14]
[278.18, 505.38, 0.96631] [301.91, 491.14, 0.98145]
예측한 턱의 좌표 :  [301.91, 513.92]


### 해야할 일

1. 코드에서 좌표 값을 언제 저장할 건지?
-> 좌표 값은 매번 뽑혀서 나옴